# Latent Altruism: Steering Cooperative Intent in LLMs
## NeurIPS 2026 — Multi-Model Experiment Runner

### How to use this notebook

1. **Set `MODEL_KEY` in Cell 2** to choose which model to run
2. Each notebook run handles **ONE model** to stay within GPU memory
3. Run the notebook once per model, archive results, then compare

### Available models

| Key | Model | Params | Strategic Layer |
|---|---|---|---|
| `qwen-32b` | Qwen2.5-32B-Instruct | 32B | L57/65 |
| `llama-8b` | Llama-3.1-8B-Instruct | 8B | L27/32 |
| `llama-70b` | Llama-3.1-70B-Instruct | 70B | L68/80 |
| `mistral-7b` | Mistral-7B-Instruct-v0.3 | 7B | L27/32 |
| `gemma-27b` | Gemma-2-27B-it | 27B | L39/46 |

### Experiment Phases

- **Phase 1:** Calibration (AllC/AllD hidden states → steering vectors)
- **Phase 2:** Baseline IPD games
- **Phase 3:** Prompt-Cooperative Baseline
- **Phase 4:** Control Vectors (Random + Orthogonal)
- **Phase 5:** Steered games — full α sweep (SV vs CAA vs RepE)
- **Phase 6:** Layer Ablation
- **Novel A:** Strategic Layer vs Last Layer Injection
- **Novel B:** Dynamic α Steering (Latent TfT)
- **Novel C:** Orthogonal Concept Erasure (OCE)
- **Novel D:** Attention Head Importance

---

## Step 1: Clone & Setup

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/trungkiet2005/steering_cooperative.git"
REPO_DIR = "/kaggle/working/steering_cooperative"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("Repository present — pulling latest...")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

os.chdir(REPO_DIR)
print(f"\nWorking directory: {os.getcwd()}")
!ls -lh experiments/*.py

In [ ]:
!pip install -q transformers accelerate bitsandbytes scipy datasets tqdm seaborn

import torch, transformers
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU detected")

## Step 2: Select Model

⚠️ **Change `MODEL_KEY` below to run a different model.**
Each notebook session runs **one model at a time** to stay within GPU memory.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CHANGE THIS to select which model to run               ║
# ║  Options: 'qwen-32b', 'llama-8b', 'llama-70b',         ║
# ║           'mistral-7b', 'gemma-27b'                     ║
# ╚══════════════════════════════════════════════════════════╝

MODEL_KEY = 'qwen-32b'

# Optional: skip specific phases to save time
SKIP_PHASES = []  # e.g. ['phase4', 'novel_d']

# Import config and show summary
import sys
sys.path.insert(0, '.')
from experiments.config import SteeringConfig, MODEL_REGISTRY

print(f"Available models: {list(MODEL_REGISTRY.keys())}")
print(f"\nSelected: {MODEL_KEY}")
cfg = SteeringConfig(MODEL_KEY)
cfg.print_summary()

## Step 3: Run All Experiment Phases

This runs the complete pipeline:
- Calibration → Baseline → Prompt Coop → Controls → Steered α-sweep → Layer Ablation
- Novel A (strategic layer) → Novel B (dynamic) → Novel C (OCE) → Novel D (heads)

Expected runtime: **~3–5 hours** on H100 for 32B models

In [ ]:
from experiments.runner import run_all_experiments

player, cfg, svs = run_all_experiments(
    model_key=MODEL_KEY,
    skip_phases=SKIP_PHASES,
)

## Step 4 (Optional): Run Benchmarks

Additional experiments addressing reviewer feedback:
- B1: WikiText-2 perplexity (standard benchmark)
- B2: Semantic invariance (X/Y relabeling)
- B3: Cross-lingual steering transfer
- B4: Scenario-based social dilemmas

In [ ]:
import numpy as np

from experiments.benchmarks import (
    compute_perplexity_standard,
    run_semantic_invariance_test,
    run_crosslingual_test,
    run_scenario_dilemma_test,
)

sv = svs[cfg.PRIMARY_LAYER]
out_dir = cfg.OUTPUT_DIR

# B1: WikiText-2 PPL
compute_perplexity_standard(player, cfg, sv, out_dir)

# B2: Semantic invariance
run_semantic_invariance_test(player, cfg, sv, out_dir)

# B3: Cross-lingual transfer
run_crosslingual_test(player, cfg, sv, out_dir)

# B4: Scenario dilemmas
run_scenario_dilemma_test(player, cfg, sv, out_dir)

## Step 5 (Optional): Layer-wise FDI Sweep

Scans ALL transformer layers to find the strategic bottleneck.
This is the experiment that confirms the ~85% depth ratio across architectures.

In [ ]:
from experiments.runner import run_fdi_sweep

fdi_values, peak_layer = run_fdi_sweep(player, cfg, cfg.OUTPUT_DIR)
print(f"\n★ Strategic Bottleneck: Layer {peak_layer} / {cfg.N_LAYERS}")
print(f"  Relative depth: {peak_layer / cfg.N_LAYERS:.3f}")

## Step 6: Verify Output Files

In [ ]:
import os

out_dir = cfg.OUTPUT_DIR

print(f"{'='*55}")
print(f"OUTPUT FILES — {cfg.model_key}")
print(f"Directory: {out_dir}")
print(f"{'='*55}")

if os.path.exists(out_dir):
    total_size = 0
    for f in sorted(os.listdir(out_dir)):
        fp = os.path.join(out_dir, f)
        if os.path.isfile(fp):
            size_kb = os.path.getsize(fp) / 1024
            total_size += size_kb
            print(f"  {f:45s} {size_kb:7.1f} KB")
    print(f"\n  Total: {total_size:.1f} KB ({total_size/1024:.1f} MB)")
else:
    print(f"  Directory not found: {out_dir}")

## Step 7: Display Key Figures

In [ ]:
from IPython.display import Image, display
import os

def show_image(path, title=""):
    if os.path.exists(path):
        print(f"\n{'─'*60}")
        print(f"  {title or os.path.basename(path)}")
        print(f"{'─'*60}")
        display(Image(filename=path, width=900))
    else:
        print(f"  [Not found: {os.path.basename(path)}]")

out = cfg.OUTPUT_DIR

show_image(f"{out}/head_importance.png",
           f"Head Importance — Layer {cfg.STRATEGIC_LAYER} ({cfg.model_key})")
show_image(f"{out}/fdi_sweep.png",
           f"Layer-wise FDI Sweep ({cfg.model_key})")
show_image(f"{out}/crosslingual_steering.png",
           f"Cross-Lingual Steering Transfer ({cfg.model_key})")
show_image(f"{out}/scenario_dilemma_test.png",
           f"Scenario-Based Social Dilemmas ({cfg.model_key})")

## Step 8: Archive Results

In [ ]:
import shutil

archive_name = f"/kaggle/working/results_{cfg.model_key}"
shutil.make_archive(
    base_name=archive_name,
    format="gztar",
    root_dir=cfg.OUTPUT_DIR,
    base_dir=".",
)
archive = archive_name + ".tar.gz"
size_mb = os.path.getsize(archive) / 1e6
print(f"Archived: {archive}  ({size_mb:.1f} MB)")
print(f"\nTo run another model, restart the kernel and change MODEL_KEY in Step 2.")